# 02K — Population genetics


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 in_gnomad vs pathogenic


In [3]:
tmp = d[['in_gnomad_bool', 'pathogenic']].dropna()
fig = px.histogram(tmp, x='in_gnomad_bool', color='pathogenic', barmode='stack', title='in_gnomad vs pathogenic')
fig.show()


## 2. 🧪 Fisher ⭐


In [4]:
tab, odds, p, ci = fisher_bool(d, 'in_gnomad_bool', 'pathogenic')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


pathogenic,False,True
in_gnomad_bool,,
False,4387,3386
True,3503,32


,value
odds_ratio,0.011836
p_value,0.000000
or_ci_low,0.008333
or_ci_high,0.016811


## 3. 📊 allele_freq histogram


In [5]:
tmp = d[['allele_freq_num']].dropna()
fig = px.histogram(tmp, x='allele_freq_num', nbins=60, title='Allele frequency histogram')
fig.show()


## 4. 📊 log allele_freq


In [6]:
tmp = d[['allele_freq_num']].dropna().copy()
tmp['log_af'] = np.log10(tmp['allele_freq_num'])
fig = px.histogram(tmp, x='log_af', nbins=60, title='log10 allele frequency histogram')
fig.show()


## 5. 📊 allele_freq vs pathogenic


In [7]:
tmp = d[['pathogenic', 'allele_freq_num']].dropna()
fig = px.box(tmp, x='pathogenic', y='allele_freq_num', points='outliers', title='Allele frequency vs pathogenic')
fig.update_yaxes(type='log')
fig.show()


## 6. 🧪 Mann–Whitney


In [8]:
g1, g0, stat, p = mann_whitney_bool(d, 'pathogenic', 'allele_freq_num')
display(pd.Series({'n_pathogenic': len(g1), 'n_non_pathogenic': len(g0), 'u_statistic': stat, 'p_value': p}).to_frame('value'))


,value
n_pathogenic,3.200000e+01
n_non_pathogenic,3.503000e+03
u_statistic,1.878600e+04
p_value,8.970534e-11


## 7. 📊 allele_freq vs phenotype


In [9]:
tmp = d[['phenotype', 'allele_freq_num']].dropna()
fig = px.box(tmp, x='phenotype', y='allele_freq_num', points='outliers', title='Allele frequency vs phenotype')
fig.update_yaxes(type='log')
fig.show()


## 8. 🧪 Kruskal


In [10]:
names, groups, stat, p = kruskal_group(d, 'phenotype', 'allele_freq_num')
display(pd.Series({'groups': ', '.join([str(x) for x in names]), 'kruskal_h': stat, 'p_value': p}).to_frame('value'))


,value
groups,"BMD, DMD"
kruskal_h,3.153345
p_value,0.075771


## 9. 📊 hemizygote histogram


In [11]:
tmp = d[['hemizygote_num']].dropna()
fig = px.histogram(tmp, x='hemizygote_num', nbins=60, title='Hemizygote count histogram')
fig.show()


## 10. 📊 homozygote histogram


In [12]:
tmp = d[['homozygote_num']].dropna()
fig = px.histogram(tmp, x='homozygote_num', nbins=60, title='Homozygote count histogram')
fig.show()


## 11. 📊 hemizygote vs phenotype


In [13]:
tmp = d[['phenotype', 'hemizygote_num']].dropna()
fig = px.box(tmp, x='phenotype', y='hemizygote_num', points='outliers', title='Hemizygote count vs phenotype')
fig.show()


## 12. 🧪 Mann–Whitney


In [14]:
tmp = d[d['phenotype'].isin(['DMD', 'BMD'])][['phenotype', 'hemizygote_num']].dropna()
g_dmd = tmp.loc[tmp['phenotype'] == 'DMD', 'hemizygote_num']
g_bmd = tmp.loc[tmp['phenotype'] == 'BMD', 'hemizygote_num']
stat, p = mannwhitneyu(g_dmd, g_bmd, alternative='two-sided')
display(pd.Series({'n_dmd': len(g_dmd), 'n_bmd': len(g_bmd), 'u_statistic': stat, 'p_value': p}).to_frame('value'))


,value
n_dmd,3031.000000
n_bmd,198.000000
u_statistic,280067.500000
p_value,0.107257


## 13. 📊 AF vs REVEL


In [15]:
tmp = d[['allele_freq_num', 'revel_num']].dropna()
fig = px.scatter(tmp, x='allele_freq_num', y='revel_num', opacity=0.5, title='AF vs REVEL')
fig.update_xaxes(type='log')
fig.show()


## 14. 🧪 Spearman


In [16]:
tmp, rho, p = spearman_xy(d, 'allele_freq_num', 'revel_num')
display(pd.Series({'n': len(tmp), 'spearman_rho': rho, 'p_value': p}).to_frame('value'))


,value
n,1488.000000
spearman_rho,-0.020155
p_value,0.437231
